Harmonization 

In [1]:
# uninstalled dependencies
%pip install sentence-transformers networkx
import re
import unicodedata
from difflib import SequenceMatcher
from math import comb
import numpy as np
import pandas as pd
import networkx as nx
import networkx.algorithms.community as nx_comm
from scipy import sparse
from sentence_transformers import SentenceTransformer


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Config

GOLD_FILE = 'PATSTAT_500_label_datav4.csv'
NAME_COL = 'comp'
GROUP_COL = 'Group'
WEIGHT_COL = None         # e.g. 'n_families' on the quantum sample; None -> every row counts 1 (gold standard)

EMBEDDING_MODEL = "Vsevolod/company-names-similarity-sentence-transformer"  # embedding model purpose-built for company names

# Pair scoring: ALL pairs are compared (no blocking)
SCORE_WEIGHTS = {"cosine": 0.55, "jaccard": 0.30, "edit": 0.15}
SCORE_FLOOR = 0.35        # pairs scoring below this are never used (Otsu floor, and no threshold below it is tried)
PAIR_BLOCK = 500          # rows scored at a time (memory only, does not change the results)

# Graph cut
LOUVAIN_RESOLUTION = 1.0
LOUVAIN_SEED = 42
SWEEP_GRID = np.arange(0.40, 0.851, 0.025)

Parte generale per le firms -> dopo vedo come adattarla quando ho anche uni e individui

### Parte di clening del nome 
va bene per tutto uni e firms 

In [3]:
FORMS = ["INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "PLC", "GMBH", "MBH",
         "AG", "KG", "SE", "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "OY", "OYJ", "AB", "AS", "ASA", "APS", "PTY", "PTE",
         "KK", "KABUSHIKI KAISHA", "SP Z OO", "OOO", "BHD", "SDN", "PVT", "LTDA"]
FORMS_RE = "|".join(sorted(FORMS, key=len, reverse=True))     # longest first: 'CORPORATION' before 'CO'
FORMS_CJK = "股份有限公司|有限责任公司|有限責任公司|有限公司|株式会社|有限会社|合同会社|주식회사|公司"
WORDS = {"UNIVERSITY": "UNIV", "UNIVERSITAET": "UNIV", "UNIVERSITAT": "UNIV", "UNIVERSITE": "UNIV", "UNIVERSIDAD": "UNIV",
         "UNIVERSITA": "UNIV", "INSTITUTE": "INST", "INSTITUT": "INST", "INSTITUTO": "INST", "TECHNOLOGY": "TECH",
         "TECHNOLOGIES": "TECH", "TECHNOLOGICAL": "TECH", "LABORATORY": "LAB", "LABORATORIES": "LAB", "LABS": "LAB",
         "CENTRE": "CENTER", "SCIENCES": "SCIENCE", "大學": "大学"}
FILLER = {"OF", "THE", "FOR", "AND", "DE", "DI", "DU", "DES", "ET", "UND"}



def clean(name):
    s = unicodedata.normalize("NFKD", unicodedata.normalize("NFKC", str(name)).upper())
    s = unicodedata.normalize("NFC", "".join(c for i, c in enumerate(s) if not (unicodedata.combining(c) and s[i - 1] < "\u0250")))
    s = re.sub(r"[▲▼?.'’]", "", s)
    latin = re.search(r"[^\x00-\x7f]([A-Z0-9][A-Z0-9 ,&\-/]+)$", s)     # CJK name + Latin version -> Latin part
    if latin and len(re.sub(r"[^A-Z]", "", latin.group(1))) >= 3:
        s = latin.group(1)
    s = re.sub(r"\bC/O\b.*$", "", s).replace("A/S", "AS").replace("&", " AND ")
    s = re.sub(rf"^({FORMS_CJK})|({FORMS_CJK})$", "", re.sub(r"[^\w ]|_", " ", s).strip())
    s = " ".join(WORDS.get(w, w) for w in s.split() if w not in FILLER) or s
    s = re.sub(rf"^(({FORMS_RE}) )+|( ({FORMS_RE}))+$", "", s).strip() or s
    return s


df = pd.read_csv(GOLD_FILE)
df['clean'] = df[NAME_COL].apply(clean)
df = df[df['clean'].str.len() > 0].copy()  # drop empty names
names = sorted(df['clean'].unique())
print(f"names: {len(names):,}  groups: {df[GROUP_COL].nunique() if GROUP_COL in df.columns else 'n/a (no gold)'}")

names: 2,916  groups: 481


### Confronto di tutte le coppie
Non propongo più coppie plausibili: con n nomi le coppie sono n(n-1)/2 (circa 4,2 milioni con 2.911 nomi) e si confrontano **tutte**. Niente blocking, quindi nessuna coppia persa in partenza e niente MinHash / HNSW / primo token da giustificare. Il blocking serve quando i nomi sono milioni (PATSTAT intero), non nel nostro campione.
Le coppie si calcolano a blocchi di righe. L'edit distance (la parte lenta) si calcola solo se lo score massimo possibile supera `SCORE_FLOOR`: siccome edit ≤ 1 è un limite esatto, non un'approssimazione, e il risultato è identico al confronto completo.

In [4]:
# STAGE 1: embeddings of the cleaned names
model = SentenceTransformer(EMBEDDING_MODEL)
emb = model.encode(names, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
print(f"names: {len(names):,}  pairs to compare: {len(names) * (len(names) - 1) // 2:,}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/92 [00:00<?, ?it/s]

names: 2,916  pairs to compare: 4,250,070


### Parte 2 creo il grafo pesato 
Collego ogni coppia trovato con un arco pesato -> peso dato in base a:
1) similarità di sbert --> semntica pura
2) jaccard --> simile le parole ma non conta ordine (similarità token)
3) non volgio essere overconfident -> metto anche un confronto lettera per lettera con peso bassino
   

In [5]:
# STAGE 2: score ALL pairs (cosine + token Jaccard + edit ratio), block by block



def pair_features(pairs, emb_dict):
    rows = []
    for a, b in pairs:
        ea, eb = emb_dict[a], emb_dict[b]
        ta, tb = set(a.split()), set(b.split())
        rows.append([
            float(np.dot(ea, eb)),
            len(ta & tb) / len(ta | tb) if (ta | tb) else 0.0,
            SequenceMatcher(None, a, b).ratio(),
            ])
    df = pd.DataFrame(rows, columns=["cosine", "jaccard", "edit"])
    df["name_a"] = [a for a, b in pairs]
    df["name_b"] = [b for a, b in pairs]
    df["score"] = sum(df[k] * w for k, w in SCORE_WEIGHTS.items())
    return df


compared 4,250,070 pairs | kept 78,620 with score >= 0.35
score distribution (kept): median=0.638  p90=0.717  max=0.995


### Taglio del grafo 
Dopo che ho il grafo connesso toglio il peso degli archi e tengo solo le sotto sezioni a cui do il nome della company con più families 

fammi un istogramma con i pesi degli archi per vedere la solgia da scelgie 
togli tutti i limiti a otsu

In [6]:
def find_threshold_otsu(scores, bins=100, floor=SCORE_FLOOR):
    """Trova la soglia ottimale che separa match e non-match minimizzando la varianza intra-classe."""
    # Consideriamo solo la fascia plausibile (>= floor) per evitare il rumore di fondo
    s = scores[scores >= floor]
    hist, bin_edges = np.histogram(s, bins=bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0

    weight1 = np.cumsum(hist)
    weight2 = np.cumsum(hist[::-1])[::-1]

    mean1 = np.cumsum(hist * bin_centers) / np.maximum(weight1, 1)
    mean2 = (np.cumsum((hist * bin_centers)[::-1]) / np.maximum(weight2[::-1], 1))[::-1]

    # Varianza inter-classe tra le due distribuzioni
    variance_between = weight1[:-1] * weight2[1:] * (mean1[:-1] - mean2[1:]) ** 2
    idx = np.argmax(variance_between)
    return float(bin_centers[idx])


# Esecuzione unsupervised (no gold labels used)
th_otsu = find_threshold_otsu(scored["score"].values)
print(f"Soglia Otsu (unsupervised): {th_otsu:.3f}")

Soglia Otsu (unsupervised): 0.566


In [9]:
# 1. Threshold = Otsu output of THIS run (no hard-coded value)
TH = th_otsu

# 2. Build the graph and find communities (Louvain does not need the number of groups)
def cluster(th, seed=LOUVAIN_SEED):
    G = nx.Graph()
    G.add_nodes_from(names)
    edges = scored[scored["score"] >= th]
    G.add_weighted_edges_from(zip(edges["name_a"], edges["name_b"], edges["score"]))
    return nx_comm.louvain_communities(G, weight="weight", resolution=LOUVAIN_RESOLUTION, seed=seed)

def to_labels(comms):
    return {n: i for i, c in enumerate(comms) for n in c}

communities = cluster(TH)

# 3. Representative name per cluster (most frequent, then shortest)
counts = df.groupby("clean")[WEIGHT_COL].sum() if WEIGHT_COL else df["clean"].value_counts()
harmonized_map = {}
for members in communities:
    best_name = min(members, key=lambda n: (-counts.get(n, 0), len(n)))
    for m in members:
        harmonized_map[m] = best_name
df["harmonized"] = df["clean"].map(harmonized_map)
print(f"clusters: {len(communities):,} | largest cluster: {max(len(c) for c in communities):,} names")

# 4. Pairwise Precision, Recall, F1 (course metric) - only if the file has a gold column
def pairs_inside(group_sizes):
    return sum(comb(int(n), 2) for n in group_sizes)

def precision_recall_f1(gold_groups, predicted_groups):
    res_df = pd.DataFrame({"gold": list(gold_groups), "pred": list(predicted_groups)})
    tp = pairs_inside(res_df.groupby(["gold", "pred"]).size())
    tp_fp = pairs_inside(res_df.groupby("pred").size())
    tp_fn = pairs_inside(res_df.groupby("gold").size())
    precision = tp / tp_fp if tp_fp else 1.0
    recall = tp / tp_fn if tp_fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"groups found": res_df["pred"].nunique(), "precision": precision, "recall": recall, "F1": f1}

if GROUP_COL in df.columns:
    print(f"Gruppi veri nel gold standard: {df[GROUP_COL].nunique()}")
    print(pd.Series(precision_recall_f1(df[GROUP_COL], df["harmonized"])).round(3))
    dn = df.drop_duplicates("clean")        # distinct cleaned names only (identical names are trivially together)
    print("distinct names only:", {k: round(v, 3) for k, v in precision_recall_f1(dn[GROUP_COL], dn["harmonized"]).items()})


clusters: 559 | largest cluster: 169 names
Gruppi veri nel gold standard: 481
groups found    559.000
precision         0.920
recall            0.861
F1                0.890
dtype: float64
distinct names only: {'groups found': 559, 'precision': 0.915, 'recall': 0.853, 'F1': 0.883}


### Sensibilità della soglia
Otsu non usa il gold. Qui si vede quanto F1 si perde rispetto alla soglia migliore (scelta sul gold stesso, quindi è un limite superiore) e quanto cambia il risultato con il seed di Louvain.

In [8]:
# Threshold sensitivity (gold only)
if GROUP_COL in df.columns:
    f1_at = lambda th, seed=LOUVAIN_SEED: precision_recall_f1(df[GROUP_COL], df["clean"].map(to_labels(cluster(th, seed))))
    sweep = pd.DataFrame([{"th": round(float(t), 3), **f1_at(t)} for t in SWEEP_GRID])
    print(sweep.round(3).to_string(index=False))
    best, at_otsu = sweep.loc[sweep["F1"].idxmax()], f1_at(th_otsu)
    print(f"Otsu {th_otsu:.3f}: F1 {at_otsu['F1']:.3f} | best on grid {best['th']:.3f}: F1 {best['F1']:.3f} | loss {best['F1'] - at_otsu['F1']:.3f}")
    seeds = [f1_at(th_otsu, s)["F1"] for s in range(10)]
    print(f"Louvain seeds 0-9 at Otsu: F1 mean {np.mean(seeds):.3f}, sd {np.std(seeds):.3f}")

   th  groups found  precision  recall    F1
0.400           322      0.848   0.958 0.900
0.425           359      0.873   0.949 0.910
0.450           393      0.885   0.942 0.912
0.475           426      0.893   0.938 0.915
0.500           459      0.901   0.925 0.913
0.525           500      0.908   0.898 0.903
0.550           531      0.907   0.881 0.894
0.575           574      0.921   0.860 0.889
0.600           617      0.928   0.836 0.880
0.625           667      0.940   0.814 0.873
0.650           728      0.944   0.760 0.842
0.675           827      0.948   0.697 0.803
0.700           974      0.946   0.602 0.736
0.725          1183      0.953   0.402 0.565
0.750          1408      0.966   0.284 0.439
0.775          1722      0.973   0.140 0.245
0.800          2073      0.972   0.059 0.112
0.825          2311      0.982   0.030 0.058
0.850          2488      0.986   0.016 0.032
Otsu 0.566: F1 0.890 | best on grid 0.475: F1 0.915 | loss 0.025
Louvain seeds 0-9 at Otsu: F1 mean 